# End-to-end feedback-GRAPE quantum memory

This notebook tests whether feedback-GRAPE can protect an **unknown qubit state** by encoding it in the cavity, applying repeated measurement-conditioned feedback, and decoding it back into the qubit.

For every stochastic trajectory:

1. sample a Haar-random qubit state
   $$
   |\psi\rangle=\alpha|0\rangle+\beta|1\rangle,
   \qquad |\alpha|^2+|\beta|^2=1,
   $$
   and initialize the cavity in vacuum;
2. apply a trainable joint qubit--cavity **encoding pulse**;
3. apply the same recurrent feedback-GRAPE step as in the state-stabilization notebook for $N$ rounds;
4. apply a trainable joint qubit--cavity **decoding pulse**;
5. maximize the fidelity between the retrieved qubit state and the original input state.

The encoder, recurrent controller, and decoder are common to all input states. The input amplitudes $\alpha$ and $\beta$ are never supplied to the controller. Both the encoder and decoder use the same 80-coefficient quadratic B-spline parameterization and the same pulse duration as one feedback correction pulse.


## Imports and reproducibility


In [14]:
import json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import trange

import jax
import jax.numpy as jnp
import jax.scipy as jsp

import optax
import flax.linen as nn
from flax.training.train_state import TrainState

from toolbox import *

print("jax", jax.__version__)
print("jaxlib", jax.lib.__version__)
print("devices", jax.devices())

seed = 1234
key = jax.random.key(seed)


jax 0.6.2
jaxlib 0.6.2
devices [CudaDevice(id=0)]


## Experimental parameters


In [15]:
def _strip_comment_keys(obj):
    # Recursively drop configuration keys that start with '//'.
    if isinstance(obj, dict):
        return {
            k: _strip_comment_keys(v)
            for k, v in obj.items()
            if not (isinstance(k, str) and k.startswith("//"))
        }
    if isinstance(obj, list):
        return [_strip_comment_keys(x) for x in obj]
    return obj


def load_config_dict(config_directory: Path, filename: str = "configuration.json") -> dict:
    path = (config_directory / filename).resolve()
    with path.open("r", encoding="utf-8") as f:
        return _strip_comment_keys(json.load(f))


config_directory = Path("")
cfg = load_config_dict(config_directory)

# Frequencies are angular frequencies in rad/us; numerically, MHz = 1/us.
chi = -2 * jnp.pi * cfg["chi_kHz"] * 1e-3
K_cg = 2 * jnp.pi * cfg["self_Kerr_kHz"] * 1e-3
K_ce = 2 * jnp.pi * cfg["self_Kerr_if_qubit_in_e_kHz"] * 1e-3

t1_qub = cfg["qubit_T1_us"]
t2_qub = cfg["qubit_T2_us"]
t1_cav = cfg["storage_T1_us"]

print("chi / 2pi (MHz):", float(chi / (2 * jnp.pi)))
print("K_c,g / 2pi (MHz):", float(K_cg / (2 * jnp.pi)))
print("K_c,e / 2pi (MHz):", float(K_ce / (2 * jnp.pi)))
print("T1 qubit (us):", t1_qub)
print("T2 qubit (us):", t2_qub)
print("T1 cavity (us):", t1_cav)


chi / 2pi (MHz): -0.6597474879999999
K_c,g / 2pi (MHz): -0.0007143000000000001
K_c,e / 2pi (MHz): -0.00035699999999999995
T1 qubit (us): 45.6
T2 qubit (us): 36.4
T1 cavity (us): 231.9


## Control parameterization and trainable parameters

The recurrent policy is unchanged: a GRU with 80 hidden features is followed by two dense layers and returns four sets of 20 B-spline coefficients for the qubit and cavity I/Q quadratures.

The complete trainable parameter tree is

$$
\theta=\{\theta_{\mathrm{RNN}},\,c_{\mathrm{enc}},\,c_{\mathrm{dec}}\},
$$

where $c_{\mathrm{enc}},c_{\mathrm{dec}}\in\mathbb{R}^{4\times20}$. The same optimizer updates all three components jointly.


In [16]:
num_bspln = 20
delta_ns = 64

inp_shape = (1,)
rec_features = 4 * num_bspln
dense_features = [2 * rec_features, 2 * rec_features]

init_lr = 2.0e-4
final_lr = 1.0e-5
constant_steps = 100
decay_steps = 10_000
pulse_init_scale = 0.04

lr_schedule = optax.join_schedules(
    schedules=[
        optax.constant_schedule(init_lr),
        optax.linear_schedule(
            init_value=init_lr,
            end_value=final_lr,
            transition_steps=decay_steps,
        ),
        optax.constant_schedule(final_lr),
    ],
    boundaries=[constant_steps, constant_steps + decay_steps],
)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_schedule),
)


class FeedbackPolicy(nn.Module):
    @nn.compact
    def __call__(self, carry, x):
        gru_cell = nn.GRUCell(features=rec_features)
        carry, x = gru_cell(carry, x)

        for n_features in dense_features:
            x = nn.Dense(n_features)(x)
            x = jax.nn.relu(x)

        x = nn.Dense(4 * num_bspln)(x)
        return carry, x.reshape(4, num_bspln)


def init_carry(key):
    return nn.GRUCell(features=rec_features).initialize_carry(key, inp_shape)


def create_train_state(key):
    key_model, key_enc, key_dec = jax.random.split(key, 3)
    model = FeedbackPolicy()

    fake_carry = init_carry(jax.random.key(0))
    fake_input = jnp.ones(inp_shape)
    controller_params = model.init(key_model, fake_carry, fake_input)["params"]

    params = {
        "controller": controller_params,
        "encoding_coeffs": pulse_init_scale * jax.random.normal(
            key_enc, (4, num_bspln)
        ),
        "decoding_coeffs": pulse_init_scale * jax.random.normal(
            key_dec, (4, num_bspln)
        ),
    }

    return TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=optimizer,
    )


def apply_feedback_policy(train_state, params, carry, measurement):
    return train_state.apply_fn(
        {"params": params["controller"]}, carry, measurement
    )


key, subkey = jax.random.split(key)
train_state = create_train_state(subkey)

print("RNN output size:", rec_features)
print("dense layers:", dense_features)
print("encoder coefficients:", train_state.params["encoding_coeffs"].shape)
print("decoder coefficients:", train_state.params["decoding_coeffs"].shape)


RNN output size: 80
dense layers: [160, 160]
encoder coefficients: (4, 20)
decoder coefficients: (4, 20)


# Physical model

The physical model below is the same as in the original cavity-state stabilization notebook.

During the feedback latency, the measured qubit has already been reset to $|0\rangle$, so the cavity evolves under its ground-state self-Kerr and photon loss. During every driven pulse, the joint qubit--cavity Hamiltonian contains the dispersive coupling and the qubit-state-dependent cavity Kerr. Qubit relaxation, qubit dephasing, and cavity relaxation are applied throughout the pulse.


## Idle evolution: cavity self-Kerr and photon loss


In [17]:
def liouv_relax_channel(c):
    # Lindblad dissipator in the row-major vectorization convention.
    d = c.shape[0]
    I = identity(d)
    cdag_c = hconj(c) @ c

    L = tensor(c, c.conj())
    L -= 0.5 * tensor(cdag_c, I)
    L -= 0.5 * tensor(I, cdag_c.T)
    return L


def liouv_hamiltonian(H):
    # Hamiltonian Liouvillian -i[H,rho] in row-major vectorization.
    d = H.shape[0]
    I = identity(d)
    return -1j * (tensor(H, I) - tensor(I, H.T))


def make_cavity_idle_matrix(N_cav, t1_cav, t_idle):
    a = destroy(N_cav)
    adag = hconj(a)

    H_kerr_cav = 0.5 * K_cg * adag @ adag @ a @ a
    c_loss = a / jnp.sqrt(t1_cav)

    L_idle = liouv_hamiltonian(H_kerr_cav)
    L_idle += liouv_relax_channel(c_loss)
    return jsp.linalg.expm(L_idle * t_idle)


def make_idle_cav(N_cav, t1_qub, t2_qub, t1_cav, t_idle):
    # t1_qub and t2_qub are intentionally unused here: the qubit has been reset.
    idle_cav_matrix = make_cavity_idle_matrix(N_cav, t1_cav, t_idle)
    ket_g = basis(2)
    rho_g = ket_g @ hconj(ket_g)

    def idle_cav(rho):
        rho_cav = rho[:N_cav, :N_cav]
        rho_cav = (idle_cav_matrix @ rho_cav.reshape(-1, 1)).reshape(
            N_cav, N_cav
        )

        rho_out = tensor(rho_g, rho_cav)
        return rho_out / jnp.trace(rho_out)

    return idle_cav


## Driven joint qubit--cavity evolution with relaxation


In [18]:
mu_qub = 20.0
mu_cav = 20.0


def build_drive_htree(N_cav, chi, e_qub, e_cav):
    a = tensor(identity(2), destroy(N_cav))
    adag = hconj(a)
    n_phot = adag @ a

    sigz = tensor(sigma.z, identity(N_cav))
    sigp = tensor(sigma.p, identity(N_cav))
    one = tensor(identity(2), identity(N_cav))

    H_disp = (chi / 2) * n_phot @ (one - sigz)

    proj_g = 0.5 * (one + sigz)
    proj_e = 0.5 * (one - sigz)
    kerr_op = adag @ adag @ a @ a
    H_kerr = 0.5 * (K_cg * proj_g + K_ce * proj_e) @ kerr_op

    H_drift = H_disp + H_kerr

    return [
        [H_drift, 1, 1, 0],
        [mu_qub * sigp, e_qub, 1, 1],
        [mu_cav * adag, 1j * jnp.conj(e_cav), 1, 1],
    ]


def get_uevol_matrix(N_cav, chi, e_qub, e_cav, dt):
    htree = build_drive_htree(N_cav, chi, e_qub, e_cav)
    return evol_hdt_exp(hmat_from_htree(htree), dt)


def liouv_relax_channel_qubit_cav(c):
    L = tensor(c, c.conj())
    L -= 0.5 * tensor(hconj(c) @ c, identity(len(c)))
    L -= 0.5 * tensor(identity(len(c)), (hconj(c) @ c).T)
    return L


def eval_idling_matrix(c_ops, t):
    L = sum(liouv_relax_channel_qubit_cav(c) for c in c_ops)
    return jsp.linalg.expm(L * t)


def build_c_ops(N_cav, t1_qub, t2_qub, t1_cav):
    t_phi_qub = 1 / ((1 / t2_qub) - (1 / t1_qub) / 2)

    sigm = tensor(sigma.m, identity(N_cav))
    sigz = tensor(sigma.z, identity(N_cav))
    a = tensor(identity(2), destroy(N_cav))

    return [
        sigm / jnp.sqrt(t1_qub),
        sigz / jnp.sqrt(2 * t_phi_qub),
        a / jnp.sqrt(t1_cav),
    ]


def make_idle_dt(N_cav, t1_qub, t2_qub, t1_cav, t):
    idling_matrix = eval_idling_matrix(
        build_c_ops(N_cav, t1_qub, t2_qub, t1_cav), t
    )

    def idle_dt(rho):
        rho_vec = idling_matrix @ rho.reshape(-1, 1)
        return rho_vec.reshape(rho.shape)

    return idle_dt


def make_drive_with_relax(N_cav, chi, t1_qub, t2_qub, t1_cav, dt_idle):
    idle_after_drive_dt = make_idle_dt(
        N_cav, t1_qub, t2_qub, t1_cav, dt_idle
    )

    def drive_with_relax(rho_init, es_qub, es_cav, dts):
        def one_interval(rho, fields_and_dt):
            e_qub, e_cav, dt = fields_and_dt
            U = get_uevol_matrix(N_cav, chi, e_qub, e_cav, dt)[0]
            rho = U @ rho @ hconj(U)
            rho = idle_after_drive_dt(rho)
            return rho, None

        rho_final, _ = jax.lax.scan(
            one_interval, rho_init, [es_qub, es_cav, dts]
        )
        return rho_final

    return drive_with_relax


def ctrl_from_coeffs(ctrl_coeffs, bsplines):
    controls = ctrl_coeffs @ bsplines
    e_qub = controls[0] + 1j * controls[1]
    e_cav = controls[2] + 1j * controls[3]
    return e_qub, e_cav


## Projective readout, cavity backaction, and qubit reset

For the sampled outcome $y\in\{0,1\}$, the corresponding diagonal qubit block is selected with its Born probability. The cavity coherences then acquire the same Gaussian phase-noise channel as in the original notebook,

$$
(\rho_c)_{mn}\longrightarrow
\exp\!\left[-\frac{\sigma_y^2}{2}(m-n)^2\right](\rho_c)_{mn},
$$

after which the qubit is reset to $|0\rangle$. The binary controller input is $m_k=+1$ for outcome $0$ and $m_k=-1$ for outcome $1$.


In [19]:
def measure_sz(key, rho_init, sigma_theta_e_deg=6.0, sigma_theta_g_deg=2.0):
    probs = rho_init.diagonal().reshape(2, -1).sum(axis=1).real
    outcome_index = jax.random.choice(key, 2, p=probs)
    prob = probs[outcome_index]

    rho_cav = jnp.where(
        outcome_index == 0,
        rho_init[: rho_init.shape[0] // 2, : rho_init.shape[1] // 2],
        rho_init[rho_init.shape[0] // 2 :, rho_init.shape[1] // 2 :],
    )

    n = jnp.arange(rho_cav.shape[0])
    mn_diff = n[:, None] - n[None, :]

    sigma_theta_deg = jnp.where(
        outcome_index == 0, sigma_theta_g_deg, sigma_theta_e_deg
    )
    sigma_theta_rad = sigma_theta_deg * jnp.pi / 180.0
    dephasing_matrix = jnp.exp(
        -0.5 * sigma_theta_rad**2 * mn_diff**2
    )
    rho_cav = dephasing_matrix * rho_cav

    ket_g = basis(2)
    rho_out = tensor(ket_g @ hconj(ket_g), rho_cav)
    rho_out /= jnp.trace(rho_out)

    measurement = 1 - 2 * outcome_index
    return measurement, prob, rho_out


## Numerical dimensions, timing, and quadratic B-spline basis


In [20]:
ndt_drive = 250
N_cav = 23
t_idle = 10.0

sigma_e_deg = 0.5
sigma_g_deg = 0.05

# Degree k=2 gives quadratic B-splines with at most three simultaneous overlaps.
k = 2
skip_left = k
skip_right = k
n_knots = num_bspln + skip_left + skip_right
n_intervals = n_knots - k

T_ns_hw = n_intervals * delta_ns
T_us_hw = T_ns_hw / 1000.0
L_ns = (k + 1) * delta_ns
t_drive = T_us_hw

assert L_ns % 4 == 0

t_drive_edges = jnp.linspace(0.0, t_drive, ndt_drive + 1)
t_drive_mids = 0.5 * (t_drive_edges[1:] + t_drive_edges[:-1])
t_drive_intvs = t_drive_edges[1:] - t_drive_edges[:-1]

bspline_builder = setup_bspline_builder(
    0.0,
    t_drive_edges[-1],
    n_knots,
    k,
    skip_left,
    skip_right,
)
bsplns_mids = bspline_builder(t_drive_mids)
bsplns_edges = bspline_builder(t_drive_edges)

print("B-spline degree:", k)
print("number of coefficients per quadrature:", num_bspln)
print("single B-spline support:", L_ns, "ns")
print("encoding/correction/decoding pulse duration:", T_ns_hw, "ns")
print("maximum simultaneous B-splines:", k + 1)


B-spline degree: 2
number of coefficients per quadrature: 20
single B-spline support: 192 ns
encoding/correction/decoding pulse duration: 1408 ns
maximum simultaneous B-splines: 3


# End-to-end quantum-memory trajectory

The initial state is

$$
\rho_0(\psi)=|\psi\rangle\!\langle\psi|\otimes|0\rangle_c\!\langle0|.
$$

The trainable encoding pulse is applied first. Each feedback step then performs, in order:

1. projective qubit measurement, effective cavity backaction, and qubit reset;
2. cavity evolution during the feedback latency;
3. GRU update from the binary measurement outcome;
4. measurement-conditioned joint qubit--cavity correction pulse.

After $N$ steps, the trainable decoding pulse is applied directly to the final joint state. No measurement or reset is inserted between the final correction and decoding. The output cavity is traced out and the terminal reward is

$$
F_{\mathrm{ret}}(\psi,\mathbf m)
=\langle\psi|\rho_{q,\mathrm{out}}|\psi\rangle.
$$


In [21]:
def sample_haar_qubit(key):
    # Sample a pure qubit uniformly from the Bloch sphere.
    key_z, key_phi = jax.random.split(key)
    z = jax.random.uniform(key_z, minval=-1.0, maxval=1.0)
    phi = 2 * jnp.pi * jax.random.uniform(key_phi)

    alpha = jnp.sqrt((1.0 + z) / 2.0)
    beta = jnp.exp(1j * phi) * jnp.sqrt((1.0 - z) / 2.0)
    psi = alpha * basis(2, 0) + beta * basis(2, 1)

    bloch = jnp.array(
        [
            2 * jnp.real(jnp.conj(alpha) * beta),
            2 * jnp.imag(jnp.conj(alpha) * beta),
            jnp.abs(alpha) ** 2 - jnp.abs(beta) ** 2,
        ]
    )
    return psi, bloch


def reduced_qubit_state(rho_joint, N_cav):
    rho_tensor = rho_joint.reshape(2, N_cav, 2, N_cav)
    return jnp.trace(rho_tensor, axis1=1, axis2=3)


def qubit_state_fidelity(rho_qubit, psi_target):
    return jnp.real((hconj(psi_target) @ rho_qubit @ psi_target)[0, 0])


def qubit_bloch_vector(rho_qubit):
    return jnp.array(
        [
            2 * jnp.real(rho_qubit[0, 1]),
            -2 * jnp.imag(rho_qubit[0, 1]),
            jnp.real(rho_qubit[0, 0] - rho_qubit[1, 1]),
        ]
    )


## One recurrent feedback step


In [22]:
def make_feedback_step(N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle):
    idle_cav = make_idle_cav(N_cav, t1_qub, t2_qub, t1_cav, t_idle)
    drive = make_drive_with_relax(
        N_cav,
        chi,
        t1_qub,
        t2_qub,
        t1_cav,
        t_drive_intvs[0],
    )

    def feedback_step(key, rho, train_state, params, carry):
        measurement, prob, rho = measure_sz(
            key,
            rho,
            sigma_theta_e_deg=sigma_e_deg,
            sigma_theta_g_deg=sigma_g_deg,
        )

        rho = idle_cav(rho)

        rnn_input = jnp.array([measurement], dtype=float)
        carry, ctrl_coeffs = apply_feedback_policy(
            train_state, params, carry, rnn_input
        )

        e_qub, e_cav = ctrl_from_coeffs(ctrl_coeffs, bsplns_mids)
        rho = drive(rho, e_qub, e_cav, t_drive_intvs)
        return rho, carry, measurement, prob

    return feedback_step


## Encoding, repeated feedback, decoding, and terminal fidelity


In [23]:
def make_quantum_memory_trajectories(
    N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
):
    feedback_step = make_feedback_step(
        N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
    )
    drive = make_drive_with_relax(
        N_cav,
        chi,
        t1_qub,
        t2_qub,
        t1_cav,
        t_drive_intvs[0],
    )

    def trajectory_from_state(
        key,
        train_state,
        params,
        num_feedback_steps,
        psi_input,
    ):
        key_carry, key_scan = jax.random.split(key)
        carry = init_carry(key_carry)

        psi_initial = tensor(psi_input, basis(N_cav, 0))
        rho = psi_initial @ hconj(psi_initial)

        e_qub, e_cav = ctrl_from_coeffs(
            params["encoding_coeffs"], bsplns_mids
        )
        rho = drive(rho, e_qub, e_cav, t_drive_intvs)

        def one_feedback_step(scan_carry, _):
            key_step, rho_step, carry_step = scan_carry
            key_step, subkey = jax.random.split(key_step)

            rho_step, carry_step, measurement, prob = feedback_step(
                subkey,
                rho_step,
                train_state,
                params,
                carry_step,
            )
            return (key_step, rho_step, carry_step), (measurement, prob)

        (_, rho, carry), (measurements, probs) = jax.lax.scan(
            one_feedback_step,
            (key_scan, rho, carry),
            length=num_feedback_steps,
        )

        e_qub, e_cav = ctrl_from_coeffs(
            params["decoding_coeffs"], bsplns_mids
        )
        rho = drive(rho, e_qub, e_cav, t_drive_intvs)

        rho_qubit_out = reduced_qubit_state(rho, N_cav)
        retrieval_fidelity = qubit_state_fidelity(
            rho_qubit_out, psi_input
        )

        return retrieval_fidelity, measurements, probs, rho_qubit_out

    def random_trajectory(
        key, train_state, params, num_feedback_steps
    ):
        key_input, key_dynamics = jax.random.split(key)
        psi_input, input_bloch = sample_haar_qubit(key_input)

        retrieval_fidelity, measurements, probs, rho_qubit_out = (
            trajectory_from_state(
                key_dynamics,
                train_state,
                params,
                num_feedback_steps,
                psi_input,
            )
        )
        return (
            retrieval_fidelity,
            measurements,
            probs,
            input_bloch,
            rho_qubit_out,
        )

    return random_trajectory, trajectory_from_state


# Feedback-GRAPE gradient

Let $\mathbf m=(m_1,\ldots,m_N)$ denote the sampled measurement record and let $P_\theta(\mathbf m|\psi)$ be its probability under the current encoder and feedback policy. The optimized objective is the average retrieval fidelity

$$
J(\theta)=
\mathbb E_{\psi\sim\mathrm{Haar}}
\mathbb E_{\mathbf m\sim P_\theta(\cdot|\psi)}
\left[F_{\mathrm{ret}}(\psi,\mathbf m)\right].
$$

The same feedback-GRAPE estimator as in the original notebook is used:

$$
\nabla_\theta J
=\mathbb E_{\psi,\mathbf m}
\left[
\nabla_\theta F_{\mathrm{ret}}
+F_{\mathrm{ret}}\nabla_\theta\log P_\theta(\mathbf m|\psi)
\right].
$$

For a fixed sampled record, automatic differentiation supplies the direct derivative through the encoder, simulated dynamics, recurrent controller, and decoder. The score-function term accounts for the dependence of the discrete measurement-record probabilities on the trainable parameters. The Haar distribution of input states is independent of $\theta$, so it requires no score term.

The code implements this gradient with the surrogate

$$
F_{\mathrm{ret}}+
\operatorname{stopgrad}(F_{\mathrm{ret}})
\sum_{k=1}^{N}\log p_\theta(m_k|m_{<k},\psi).
$$


In [24]:
def make_trajectory_with_rewgrad(
    N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
):
    random_trajectory, _ = make_quantum_memory_trajectories(
        N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
    )

    def trajectory_with_rewgrad(key, train_state, num_feedback_steps):
        def objective_for_one_sample(params):
            trajectory_output = random_trajectory(
                key,
                train_state,
                params,
                num_feedback_steps,
            )
            retrieval_fidelity, _, probs, _, _ = trajectory_output

            reward = jnp.clip(jnp.real(retrieval_fidelity), 0.0, 1.0)
            safe_probs = jnp.clip(probs, 1e-8, 1.0)
            log_record_probability = jnp.log(safe_probs).sum()

            feedback_grape_surrogate = (
                reward
                + jax.lax.stop_gradient(reward)
                * log_record_probability
            )
            return feedback_grape_surrogate, trajectory_output

        gradient_function = jax.grad(
            objective_for_one_sample, has_aux=True
        )
        reward_gradient, trajectory_output = gradient_function(
            train_state.params
        )
        return reward_gradient, trajectory_output

    return trajectory_with_rewgrad


## Vectorized training and testing batches


In [25]:
def make_train_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    batch_size,
    num_feedback_steps,
):
    trajectory_with_rewgrad = make_trajectory_with_rewgrad(
        N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
    )
    batched_trajectory_with_rewgrad = jax.vmap(
        trajectory_with_rewgrad, in_axes=(0, None, None)
    )

    @jax.jit
    def train_on_batch(key, train_state):
        keys = jax.random.split(key, batch_size)
        reward_gradients, trajectory_output = (
            batched_trajectory_with_rewgrad(
                keys, train_state, num_feedback_steps
            )
        )

        loss_gradient = jax.tree.map(
            lambda x: -x.mean(axis=0), reward_gradients
        )
        train_state = train_state.apply_gradients(
            grads=loss_gradient
        )
        return train_state, trajectory_output

    return train_on_batch


def make_test_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    batch_size,
    num_feedback_steps,
):
    random_trajectory, _ = make_quantum_memory_trajectories(
        N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
    )
    batched_random_trajectory = jax.vmap(
        random_trajectory, in_axes=(0, None, None, None)
    )

    @jax.jit
    def test_on_batch(key, train_state):
        keys = jax.random.split(key, batch_size)
        return batched_random_trajectory(
            keys,
            train_state,
            train_state.params,
            num_feedback_steps,
        )

    return test_on_batch


# Train the end-to-end memory

The default uses the same 18 feedback steps and batch size 32 as the original training notebook. Each batch contains independent Haar-random inputs and independent measurement records. The two additional joint pulses increase the simulated trajectory length from $N$ driven pulses to $N+2$ driven pulses.

The objective shown during training is the mean terminal retrieval fidelity, not an average of intermediate cavity fidelities.


In [ ]:
num_feedback_steps_train = 3
batch_size = 128
num_training_batches = 2 * constant_steps + decay_steps
print_every = 10

train_on_batch = make_train_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    batch_size,
    num_feedback_steps_train,
)

mean_retrieval_fidelity_train3 = []

progress = trange(num_training_batches)
for batch_index in progress:
    key, subkey = jax.random.split(key)
    train_state, trajectory_output = train_on_batch(
        subkey, train_state
    )

    retrieval_fidelities = trajectory_output[0]
    mean_fidelity = float(jnp.mean(retrieval_fidelities))
    mean_retrieval_fidelity_train3.append(mean_fidelity)

    if batch_index % print_every == 0:
        progress.set_postfix(
            retrieval_fidelity=f"{mean_fidelity:.5f}"
        )


  8%|▊         | 812/10200 [28:35<5:28:42,  2.10s/it, retrieval_fidelity=0.94919]

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2.8))

training_fidelity3 = np.asarray(mean_retrieval_fidelity_train3)
ax.plot(1.0 - training_fidelity3, lw=1.0)
ax.axhline(
    1.0 - 2.0 / 3.0,
    color="0.45",
    ls="--",
    lw=1.0,
    label="classical average-fidelity bound",
)
ax.set_xlabel(f"Training batch (size {batch_size})")
ax.set_ylabel("Retrieval infidelity with 3 steps of feedback")
ax.set_yscale("log")
ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
num_feedback_steps_train = 6
batch_size = 64
num_training_batches = 2 * constant_steps + decay_steps
print_every = 10

train_on_batch = make_train_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    batch_size,
    num_feedback_steps_train,
)

mean_retrieval_fidelity_train6 = []

progress = trange(num_training_batches)
for batch_index in progress:
    key, subkey = jax.random.split(key)
    train_state, trajectory_output = train_on_batch(
        subkey, train_state
    )

    retrieval_fidelities = trajectory_output[0]
    mean_fidelity = float(jnp.mean(retrieval_fidelities))
    mean_retrieval_fidelity_train6.append(mean_fidelity)

    if batch_index % print_every == 0:
        progress.set_postfix(
            retrieval_fidelity=f"{mean_fidelity:.5f}"
        )


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2.8))

training_fidelity6 = np.asarray(mean_retrieval_fidelity_train6)
ax.plot(1.0 - training_fidelity6, lw=1.0)
ax.axhline(
    1.0 - 2.0 / 3.0,
    color="0.45",
    ls="--",
    lw=1.0,
    label="classical average-fidelity bound",
)
ax.set_xlabel(f"Training batch (size {batch_size})")
ax.set_ylabel("Retrieval infidelity with 6 steps of feedback")
ax.set_yscale("log")
ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
num_feedback_steps_train = 12
batch_size = 32
num_training_batches = 2 * constant_steps + decay_steps
print_every = 10

train_on_batch = make_train_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    batch_size,
    num_feedback_steps_train,
)

mean_retrieval_fidelity_train12 = []

progress = trange(num_training_batches)
for batch_index in progress:
    key, subkey = jax.random.split(key)
    train_state, trajectory_output = train_on_batch(
        subkey, train_state
    )

    retrieval_fidelities = trajectory_output[0]
    mean_fidelity = float(jnp.mean(retrieval_fidelities))
    mean_retrieval_fidelity_train12.append(mean_fidelity)

    if batch_index % print_every == 0:
        progress.set_postfix(
            retrieval_fidelity=f"{mean_fidelity:.5f}"
        )


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 2.8))

training_fidelity12 = np.asarray(mean_retrieval_fidelity_train12)
ax.plot(1.0 - training_fidelity12, lw=1.0)
ax.axhline(
    1.0 - 2.0 / 3.0,
    color="0.45",
    ls="--",
    lw=1.0,
    label="classical average-fidelity bound",
)
ax.set_xlabel(f"Training batch (size {batch_size})")
ax.set_ylabel("Retrieval infidelity with 12 steps of feedback")
ax.set_yscale("log")
ax.legend(frameon=False)
fig.tight_layout()


# Held-out validation

For uniformly distributed unknown pure qubit states, any measure-and-prepare strategy is limited to an average fidelity of $2/3$. Exceeding this value on held-out Haar-random states is therefore the first meaningful indication that the learned channel preserves quantum information rather than only classical state information.


In [ ]:
num_feedback_steps_test = num_feedback_steps_train
test_batch_size = 128

test_on_batch = make_test_on_batch(
    N_cav,
    chi,
    t1_qub,
    t2_qub,
    t1_cav,
    t_idle,
    test_batch_size,
    num_feedback_steps_test,
)

key, subkey = jax.random.split(key)
test_output = test_on_batch(subkey, train_state)

test_fidelities = np.asarray(test_output[0])
test_mean = test_fidelities.mean()
test_sem = test_fidelities.std(ddof=1) / np.sqrt(test_batch_size)

print(f"Held-out Haar-average retrieval fidelity: {test_mean:.5f} +/- {test_sem:.5f}")
print(f"Minimum held-out fidelity: {test_fidelities.min():.5f}")
print(f"Fraction above 2/3: {np.mean(test_fidelities > 2/3):.3f}")

fig, ax = plt.subplots(figsize=(5.0, 2.8))
ax.hist(test_fidelities, bins=20, color="#6f3db5", alpha=0.8)
ax.axvline(2 / 3, color="0.25", ls="--", label="classical bound")
ax.set_xlabel("Retrieval fidelity")
ax.set_ylabel("Held-out states")
ax.legend(frameon=False)
fig.tight_layout()


## Six cardinal input states

The six Pauli eigenstates test the poles of all three Bloch-sphere axes. Each reported value is averaged over independent stochastic measurement records while keeping the input state fixed.


In [ ]:
ket_0 = basis(2, 0)
ket_1 = basis(2, 1)

cardinal_states = [
    ket_0,
    ket_1,
    (ket_0 + ket_1) / jnp.sqrt(2),
    (ket_0 - ket_1) / jnp.sqrt(2),
    (ket_0 + 1j * ket_1) / jnp.sqrt(2),
    (ket_0 - 1j * ket_1) / jnp.sqrt(2),
]
cardinal_labels = ["+Z", "-Z", "+X", "-X", "+Y", "-Y"]
shots_per_cardinal_state = 64

_, trajectory_from_state = make_quantum_memory_trajectories(
    N_cav, chi, t1_qub, t2_qub, t1_cav, t_idle
)


@jax.jit
def evaluate_one_fixed_state(keys, psi_input, train_state):
    return jax.vmap(
        lambda one_key: trajectory_from_state(
            one_key,
            train_state,
            train_state.params,
            num_feedback_steps_test,
            psi_input,
        )
    )(keys)


cardinal_fidelities = []
cardinal_output_bloch = []

for label, psi_input in zip(cardinal_labels, cardinal_states):
    key, subkey = jax.random.split(key)
    state_keys = jax.random.split(subkey, shots_per_cardinal_state)
    state_output = evaluate_one_fixed_state(
        state_keys, psi_input, train_state
    )

    mean_fidelity = float(jnp.mean(state_output[0]))
    mean_rho_qubit = jnp.mean(state_output[3], axis=0)

    cardinal_fidelities.append(mean_fidelity)
    cardinal_output_bloch.append(
        np.asarray(qubit_bloch_vector(mean_rho_qubit))
    )
    print(
        f"{label}: fidelity={mean_fidelity:.5f}, "
        f"output Bloch={cardinal_output_bloch[-1]}"
    )

fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.bar(cardinal_labels, cardinal_fidelities, color="#4b7fe8")
ax.axhline(2 / 3, color="0.25", ls="--", label="classical bound")
ax.set_ylim(0, 1)
ax.set_ylabel("Retrieval fidelity")
ax.legend(frameon=False)
fig.tight_layout()


# Save the jointly trained controller, encoder, and decoder


In [ ]:
import orbax.checkpoint as ocp

save_root = Path("fg_trained/quantum_memory")
timestamp = datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M")
run_folder = save_root.resolve() / timestamp
checkpoint_path = run_folder / "params_only" / "1"
run_folder.mkdir(parents=True, exist_ok=True)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

checkpointer = ocp.Checkpointer(ocp.StandardCheckpointHandler())
checkpointer.save(
    checkpoint_path,
    args=ocp.args.StandardSave(train_state.params),
)

params_host = jax.device_get(train_state.params)
encoding_coeffs = np.asarray(params_host["encoding_coeffs"])
decoding_coeffs = np.asarray(params_host["decoding_coeffs"])

np.savez(
    run_folder / "memory_training_results.npz",
    mean_retrieval_fidelity_train=np.asarray(
        mean_retrieval_fidelity_train
    ),
    held_out_retrieval_fidelities=test_fidelities,
    cardinal_fidelities=np.asarray(cardinal_fidelities),
    cardinal_output_bloch=np.asarray(cardinal_output_bloch),
    encoding_coeffs=encoding_coeffs,
    decoding_coeffs=decoding_coeffs,
    encoding_waveforms=encoding_coeffs @ np.asarray(bsplns_edges),
    decoding_waveforms=decoding_coeffs @ np.asarray(bsplns_edges),
)

metadata = {
    "seed": seed,
    "num_feedback_steps_train": num_feedback_steps_train,
    "batch_size": batch_size,
    "num_training_batches": num_training_batches,
    "N_cav": N_cav,
    "pulse_duration_ns": int(T_ns_hw),
    "num_bsplines": num_bspln,
    "held_out_mean_fidelity": float(test_mean),
    "held_out_sem": float(test_sem),
}
with (run_folder / "metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved to:", run_folder)
